# A/B: pipeline gốc  vs  pipeline + UOTMột notebook, hai lần train **khác nhau đúng một biến**: `--use-uot`.| | ||---|---|| **A. Baseline** | pipeline MMA-DFER gốc, không sửa một dòng nào || **B. UOT** | y hệt A, thêm `--use-uot` |Cùng checkpoint xuất phát, cùng seed (`seed=1` hardcode), cùng fold, cùng epoch, cùng lr.Hiệu số `B − A` chính là đóng góp của UOT.UOT đụng vào đâu trong pipeline gốc: xem `UOT_DIFF.md` (3 điểm chạm trong`Generate_Model.py`, dataloader và hai encoder nguyên vẹn).

## 1. Clone

In [ ]:
WORK = "/kaggle/working/DEFR-UOT"import os, shutilif os.path.exists(WORK):    shutil.rmtree(WORK)!git clone -b feat/uot-fusion --depth 1 https://github.com/YouttyLe-DSAI/DEFR-UOT.git {WORK}%cd {WORK}!git log --oneline -1

## 2. CONFIG — cell duy nhất cần sửa

In [ ]:
DATASET = "MAFW"                 # "MAFW" hoặc "DFEW"DATA    = "/kaggle/temp/data"MODEL_DIR = "/kaggle/input/models/tunalmt/modelmma/pytorch/default/1"CKPT      = f"{MODEL_DIR}/checkpoint"RESUME    = f"{CKPT}/{DATASET}_224/fold{{fold}}_224.pth"   # {{fold}} do main.py tự thay# Warm-start từ checkpoint đã hội tụ -> 5 epoch là đủ, và lr phải nhỏ.# Train từ đầu cần 25 epoch (~10-20h/fold), không vừa giới hạn 12h của Kaggle.EPOCHS, BATCH, LR, WORKERS, FOLD = 5, 4, 2e-5, 2, 1FRAMES_ROOT = f"{DATA}/mfaw/clips_faces" if DATASET == "MAFW" else f"{DATA}/dfew/clip_224x224"print(DATASET, "| frames root:", FRAMES_ROOT)print("resume     :", RESUME)

## 3. Dependencies`timm==0.9.16` bắt buộc — image Kaggle dùng timm 1.x, mà `models/models_vit.py` kế thừa`timm.models.vision_transformer.VisionTransformer`, API đổi giữa hai major version.Không cài đè torch (mất bản CUDA của Kaggle).

In [ ]:
!pip install -q timm==0.9.16 einops==0.7.0 librosa==0.10.1import torch, timmprint("torch", torch.__version__, "| timm", timm.__version__,      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 4. Dựng dữ liệuDataloader suy ra đường dẫn `.wav` **từ đường dẫn frame bằng phép thay chuỗi**, mà trênKaggle các shard nằm ở những mount khác nhau. Cell này dựng cây symlink đúng layoutloader mong đợi — không sửa một dòng nào của code gốc.Kiểm tra output: `LAYOUT OK`, và `clips w/o wav` phải nhỏ (MAFW) / bằng 0 (DFEW).

In [ ]:
!python tools/kaggle_setup.py --dataset {DATASET} --auto --out {DATA}

## 5. Trỏ annotationĐiều kiện đi tiếp: `missing folder 0`, `EMPTY folder 0`, **`UNMATCHED path 0`**,`labels seen` = `0..10` (MAFW) / `0..6` (DFEW).

In [ ]:
!cp -r annotation annotation.bak!python tools/retarget_annotations.py --dataset {DATASET} --new-root {FRAMES_ROOT} \    --recount --drop-missing!python tools/check_data.py --annotation annotation/{DATASET}_set_{FOLD}_train{'_faces' if DATASET=='MAFW' else ''}.txt --n 300

## 6. Checkpoint encoderHai file này chỉ dùng lúc **dựng** model; `--resume` sẽ ghi đè trọng số ngay sau đó.Nhưng thiếu chúng thì `GenerateModel.__init__` chết ở `torch.load`.

In [ ]:
!cp {CKPT}/pretrained.pth ./audiomae_pretrained.pth!cp {CKPT}/mae_face_visualize_vit_base.pth ./mae_face_pretrain_vit_base.pth!ls -la *.pth

## 7. Smoke test — 1 phút, đừng bỏ quaCần thấy:- **`audio liveness ... 0/N dead`** — `N/N dead` nghĩa là các `.wav` không được tìm thấy  và audio đang là hằng số. Dừng lại.- **`max|baseline - uot| = 0.000e+00  OK`** — dựng model có UOT rồi tắt cờ thì output  trùng khít bản gốc. Đây là bằng chứng UOT không phá pipeline gốc.- gradient của **gate** khác 0 sau vài bước (`proj`/`norm` bằng 0 ở step 0 là đúng:  chúng nằm sau `tanh(gate)=0`)- `peak GPU memory` — cho biết `BATCH` nào vừa VRAM

In [ ]:
!python tools/smoke_test.py --dataset {DATASET} --use-uot --batch-size 2

## 8. NHÁNH A — pipeline gốcKhông có `--use-uot`. Đây là MMA-DFER nguyên bản.**Soi ngay khi chạy:**```Resumed from .../fold1_224.pth  missing (OTHER -- should be 0): 0      <- khác 0 = checkpoint không khớpEpoch: [0][0/1833]  Loss 0.70xx  Accuracy 75.000```**Loss step 0 phải ~0.7.** Ra ~2.9 nghĩa là warm-start không ăn (classifier đang ngẫunhiên) — dừng ngay, đừng để chạy 4 giờ.Thời gian: ~45–50 phút/epoch → **~4 giờ**.

In [ ]:
!python main.py --dataset {DATASET} --folds {FOLD} --epochs {EPOCHS} \  --batch-size {BATCH} --workers {WORKERS} --lr {LR} --weight-decay 1e-2 \  --print-freq 20 --temporal-layers 1 --img-size 224 \  --resume "{RESUME}" --exper-name AB_BASE

## 9. NHÁNH B — pipeline + UOTKhác nhánh A **đúng một thứ**: `--use-uot` và các tham số của nó.Vì gate khởi tạo bằng 0, cell này **phải bắt đầu từ đúng loss như nhánh A**. Khác đi làdấu hiệu UOT phá gì đó ngay lúc khởi tạo — kiểm tra miễn phí, tận dụng đi.

In [ ]:
!python main.py --dataset {DATASET} --folds {FOLD} --epochs {EPOCHS} \  --batch-size {BATCH} --workers {WORKERS} --lr {LR} --weight-decay 1e-2 \  --print-freq 20 --temporal-layers 1 --img-size 224 \  --resume "{RESUME}" \  --use-uot --uot-eps 0.05 --uot-tau 1.0 --uot-iters 10 \  --exper-name AB_UOT

## 10. So sánh

In [ ]:
import glob, rerows = []for log in sorted(glob.glob("log/*/log.txt")):    txt  = open(log).read()    name = log.split("/")[1]    uar  = re.findall(r"UAR: ([\d.]+)", txt)    war  = re.findall(r"WAR: ([\d.]+)", txt)    accs = re.findall(r"Current Accuracy: ([\d.]+)", txt)    ep   = re.findall(r"An epoch time: ([\d.]+)", txt)    if uar and war:        rows.append((name, float(uar[-1]), float(war[-1])))    print(f"{name}")    print(f"   val acc mỗi epoch : {accs}")    print(f"   UAR / WAR         : {uar} / {war}")    if ep:        print(f"   epoch trung bình  : {sum(map(float,ep))/len(ep)/60:.1f} phút")    print()base = [r for r in rows if "AB_BASE" in r[0]]uot  = [r for r in rows if "AB_UOT"  in r[0]]if base and uot:    print("=" * 46)    print(f"  A (gốc)  UAR {base[0][1]:.2f}   WAR {base[0][2]:.2f}")    print(f"  B (UOT)  UAR {uot[0][1]:.2f}   WAR {uot[0][2]:.2f}")    print(f"  đóng góp của UOT : {uot[0][1]-base[0][1]:+.2f} UAR   {uot[0][2]-base[0][2]:+.2f} WAR")    print("=" * 46)    print("  Một fold, 5 epoch warm-start -> đây là chỉ báo, chưa phải kết luận.")    print("  Fold spread trên bộ này tới ~12 điểm UAR; cần đủ 5 fold mới kết luận được.")

## Ghi chú- Hai nhánh ~8 giờ. Vừa một phiên 12 giờ, nhưng **an toàn hơn là mỗi nhánh một phiên**  (mỗi phiên có 12 giờ riêng). Dùng **Save Version → Save & Run All (Commit)**; phiên  tương tác bị ngắt khi idle.- `/kaggle/temp` bị xoá sau mỗi phiên → chạy lại Cell 4 mỗi lần mở notebook.- Kết quả này trả lời *"UOT có cải thiện thêm trên một model đã hội tụ không"*. Muốn  khẳng định UOT tốt hơn **khi train từ đầu** thì phải bỏ `--resume`, chạy đủ 25 epoch  × 5 fold — việc đó cần server, Kaggle không đủ.